# Mistral 7B + QLoRA — Google Colab

**Окружение:** Google Colab Free, GPU T4 16 GB.  
**Метод:** QLoRA (4-bit NF4 + LoRA-адаптеры).  
**Полные 43к примеров обучения, 3 эпохи** — для прямого сравнения с Llama QLoRA.

### Перед запуском:
1. **Runtime → Change runtime type → T4 GPU**
2. **Слева 🔑 (Secrets)** → New secret → `HF_TOKEN` = ваш токен с https://huggingface.co/settings/tokens  
   (включите тумблер «Notebook access»)
3. Примите лицензию Mistral на https://huggingface.co/mistralai/Mistral-7B-v0.1

### Порядок запуска:
1. **Блок 0** → ждать установки → **Runtime → Restart session**
2. После рестарта запускать с Блока 1 подряд до Блока 11

## ⚠️ Сначала фикс bitsandbytes

In [ ]:
# Блок 0. Свежая bitsandbytes (без сломанной triton.ops зависимости)
# ВАЖНО: после этого блока обязательно Runtime → Restart session, потом запускайте дальше
!pip uninstall -y bitsandbytes
!pip install -q --no-cache-dir --no-deps "bitsandbytes>=0.45.0,<0.47.0"
print("✅ bitsandbytes установлена. ТЕПЕРЬ: Runtime → Restart session, затем запустите Блок 1.")


## После Restart Session ↓ — запускайте все ячейки подряд

### Установка и импорты

In [ ]:
# Блок 1. Установка остальных библиотек и импорты
!pip install -q -U transformers peft accelerate datasets evaluate scikit-learn

import os, time, gc, json
from pathlib import Path
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    BitsAndBytesConfig, TrainingArguments, Trainer,
    DataCollatorWithPadding, set_seed,
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, TaskType
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, hamming_loss

SEED = 42
set_seed(SEED)

# Проверка версии bitsandbytes
import bitsandbytes
print("bitsandbytes:", bitsandbytes.__version__)

print("PyTorch:", torch.__version__)
print("CUDA доступна:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")


### Логин в Hugging Face

In [ ]:
# Блок 2. Логин в Hugging Face через Colab Secrets
# В Colab: слева панель ключика 🔑 → добавьте secret с именем `HF_TOKEN`,
# значение — ваш токен с https://huggingface.co/settings/tokens
from google.colab import userdata
from huggingface_hub import login

HF_TOKEN = userdata.get("HF_TOKEN")
login(token=HF_TOKEN)
print("✅ HF auth OK")


### Параметры эксперимента

In [ ]:
# Блок 3. Параметры эксперимента
MODEL_NAME = "mistralai/Mistral-7B-v0.1"
RUN_NAME   = "mistral7b_qlora_colab"


### Данные

In [ ]:
# Блок 4. Загрузка GoEmotions и multi-hot кодирование
dataset = load_dataset("go_emotions", "simplified")
print(dataset)

label_names = dataset['train'].features['labels'].feature.names
num_labels = len(label_names)
print(f"Классов: {num_labels}")

def multi_hot(batch):
    arr = np.zeros((len(batch['text']), num_labels), dtype=np.float32)
    for i, lbls in enumerate(batch['labels']):
        arr[i, lbls] = 1.0
    return {"encoded_labels": arr.tolist()}

encoded = dataset.map(multi_hot, batched=True, remove_columns=['labels', 'id'])


### Токенизация

In [ ]:
# Блок 5. Токенизация
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

MAX_LEN = 128

def tokenize_fn(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)

tokenized = encoded.map(tokenize_fn, batched=True, remove_columns=['text'])
tokenized = tokenized.rename_column("encoded_labels", "labels")
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
print(tokenized)


### Загрузка модели в 4-bit

In [ ]:
# Блок 6. Загрузка модели в 4-bit NF4 (это и есть QLoRA)
# Colab T4 не поддерживает bf16, используем fp16
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

print(f"Загрузка {MODEL_NAME} в 4-bit NF4 ...")
t0 = time.time()
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    problem_type="multi_label_classification",
    quantization_config=bnb_config,
    device_map="auto",
    token=HF_TOKEN,
)
print(f"Модель загружена за {time.time()-t0:.1f} сек")

model.config.pad_token_id = tokenizer.pad_token_id
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

torch.cuda.empty_cache()
mem = torch.cuda.memory_allocated() / 1024**3
print(f"VRAM после загрузки: {mem:.2f} GB")


### LoRA

In [ ]:
# Блок 7. LoRA-адаптеры (те же параметры, что у Llama для честного сравнения)
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    modules_to_save=["score"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()


### Метрики

In [ ]:
# Блок 8. Метрики
THRESHOLD = 0.5

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))
    preds = (probs >= THRESHOLD).astype(int)
    labels = labels.astype(int)
    metrics = {
        "f1_macro":        f1_score(labels, preds, average="macro", zero_division=0),
        "f1_micro":        f1_score(labels, preds, average="micro", zero_division=0),
        "f1_weighted":     f1_score(labels, preds, average="weighted", zero_division=0),
        "precision_macro": precision_score(labels, preds, average="macro", zero_division=0),
        "recall_macro":    recall_score(labels, preds, average="macro", zero_division=0),
        "hamming_loss":    hamming_loss(labels, preds),
    }
    try:
        metrics["roc_auc_macro"] = roc_auc_score(labels, probs, average="macro")
    except ValueError:
        metrics["roc_auc_macro"] = float("nan")
    return metrics


### Обучение

In [ ]:
# Блок 9. Обучение
OUTPUT_DIR = f"/content/outputs/{RUN_NAME}"
os.makedirs(OUTPUT_DIR, exist_ok=True)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=8,           # на полной модели в NF4 batch 8 свободно поместится
    per_device_eval_batch_size=16,
    gradient_accumulation_steps=4,           # эффективный batch = 32 (как у Llama QLoRA)
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,
    optim="adamw_torch",                     # на T4 paged_adamw_8bit зависает!
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    seed=SEED,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    processing_class=tokenizer,              # transformers ≥5.0
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

torch.cuda.reset_peak_memory_stats()
t0 = time.time()
train_result = trainer.train()
train_time = time.time() - t0
peak_mem = torch.cuda.max_memory_allocated() / 1024**3

print(f"\n⏱  Время обучения: {train_time/60:.2f} мин")
print(f"📈 Пиковая VRAM:    {peak_mem:.2f} GB")


### Оценка на тесте

In [ ]:
# Блок 10. Финальная оценка на тестовой выборке
test_metrics = trainer.evaluate(tokenized["test"], metric_key_prefix="test")
print("\n=== Тестовые метрики ===")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k:30s}: {v:.4f}")
    else:
        print(f"{k:30s}: {v}")


### Сохранение

In [ ]:
# Блок 11. Сохранение адаптера и summary.json
adapter_dir = f"{OUTPUT_DIR}/final_adapter"
trainer.save_model(adapter_dir)
tokenizer.save_pretrained(adapter_dir)

def folder_size_mb(path):
    return sum(f.stat().st_size for f in Path(path).rglob("*") if f.is_file()) / 1024**2

adapter_size = folder_size_mb(adapter_dir)
print(f"💾 Размер адаптера: {adapter_size:.2f} MB")

summary = {
    "run_name": RUN_NAME,
    "model": MODEL_NAME,
    "method": "QLoRA (4-bit NF4)",
    "lora_r": 16,
    "lora_alpha": 32,
    "epochs": training_args.num_train_epochs,
    "effective_batch": training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps,
    "learning_rate": training_args.learning_rate,
    "train_time_min": round(train_time/60, 2),
    "peak_vram_gb": round(peak_mem, 2),
    "adapter_size_mb": round(adapter_size, 2),
    "test_metrics": {k: float(v) for k, v in test_metrics.items() if isinstance(v, (int, float))},
}
with open(f"{OUTPUT_DIR}/summary.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("\n=== ИТОГ ===")
print(json.dumps(summary, indent=2, ensure_ascii=False))
print(f"\n📁 Файлы в: {OUTPUT_DIR}")

# Полезно: скачайте summary.json себе
from google.colab import files
files.download(f"{OUTPUT_DIR}/summary.json")
